# 3장 1강: 수집 데이터의 품질 확인 — 실습문제


## 실습 목표

- CSV의 행 수·컬럼명·자료형·실제 값을 확인하고 예상 구조와 비교할 수 있다.
- 확인용 복사본에서 결측값·빈 문자열·공백 문자열을 함께 탐색할 수 있다.
- 본문과 작성자를 기준으로 중복 후보를 찾고 후보 행 수를 해석할 수 있다.
- 수집된 데이터의 상태와 전체 수집 완료 여부를 구분할 수 있다.
- 분석 목적에 필요한 필드의 존재 여부로 활용 가능 건수·비율을 계산할 수 있다.

## 실습 환경 / 데이터

- 기준 교안: **3장 1강: 수집 데이터의 품질 확인**
- 환경: Python 3, Jupyter Notebook 또는 Google Colab
- 사용 모듈: `pandas`, Python 기본 모듈 `pathlib`
- 입력 파일: **2장 1강 API 실습에서 생성한 `quotes_raw.csv`**
- 노트북의 현재 작업 폴더에 입력 파일을 두고 실행합니다. Colab에서는 해당 실행 환경에 파일을 올린 후 진행합니다.
- 이번 실습에서는 웹사이트에 새로 요청하거나 CSV를 새로 저장하지 않습니다. 원본 파일과 원본 DataFrame을 유지하며 상태만 확인합니다.

| 문제 | 실습 내용 | 제출 결과 |
|---|---|---|
| 필수 1 | 기본 구조·예상 컬럼·출처별 건수 확인 | 코드·출력·구조 확인표·해석 답변 |
| 필수 2 | 결측·빈 문자열·중복 후보 탐색 | 코드·출력·품질 확인표·해석 답변 |
| 과제 1 | 분석 목적별 활용 가능 건수·비율, 수집 실패 판단 | 코드·출력·분석 목적별 표·사례 답변 |

| 컬럼 | 교안에서의 역할 |
|---|---|
| `text` | 명언 본문, 텍스트 분석 대상 |
| `author` | 작성자별 분석 |
| `tags` | 태그별 분석 |
| `source_url` | 수집 출처 확인 |

한 행은 명언 하나입니다. 실제 데이터에 문제가 없으면 **0건이라는 결과 그대로** 기록합니다. 문제에 제시된 가정 사례는 실제 수집 결과와 구분합니다. 중복 제거·결측 처리·정제 파일 생성은 이번 실습의 범위가 아닙니다.


## 실습 전 개념 정리

이번 강의의 질문은 **“저장된 명언을 어떤 분석에 사용할 수 있는가?”**입니다. 먼저 표의 구조를 보고, 각 칸의 값과 중복을 확인한 뒤, 분석에 필요한 조건을 적용합니다. 수집 과정이 끝까지 정상적으로 진행되었는지도 별도로 확인합니다.

| 개념 | 기초 설명 | 이번 실습에서 확인할 것 |
|---|---|---|
| CSV와 DataFrame | CSV는 파일로 저장된 표이고, DataFrame은 Python에서 읽어 다루는 표 | 파일을 읽어 만든 df의 상태 |
| 행과 컬럼 | 한 행은 명언 하나, 컬럼은 본문·작성자처럼 정보의 종류 | 행 수와 필요한 컬럼 이름 |
| Series | DataFrame에서 한 컬럼을 선택했을 때 얻는 한 줄의 값 목록 | df["text"]의 각 행에 본문이 있는지 |
| 컬럼 누락 | 필요한 정보의 이름 자체가 표에 없음 | text 컬럼이 없으면 본문 전체를 선택할 수 없음 |
| 값 누락 | 컬럼은 있지만 특정 행의 내용이 없음 | author 컬럼이 있어도 일부 작성자는 비어 있을 수 있음 |
| 결측값과 빈 문자열 | pd.NA·None·NaN 같은 결측 표시와, 내용이 없는 문자열은 구분됨 | isna()와 빈 문자열 확인을 함께 사용 |
| 공백 문자열 | 공백 문자만 들어 있어 겉으로는 빈칸처럼 보이는 문자열 | 양끝 공백을 제거한 뒤 빈 문자열인지 확인 |
| 중복 후보 | 이번 기준인 본문과 작성자가 같은 행들의 모음 | 실제 반복 수집인지 확인한 뒤 처리 방향 판단 |
| 조건표 또는 마스크 | 각 행이 조건을 만족하는지 True/False로 나타낸 결과 | 조건을 만족하는 행 선택 또는 개수 집계 |
| 분석 목적별 활용 가능 건수 | 그 분석에 필요한 값이 있는 행의 수 | 중복 처리 후 최종 건수와 구분 |
| 수집 완료 | 필요한 페이지를 정상적으로 처리하고 종료 조건에 도달한 상태 | 현재 CSV뿐 아니라 페이지별 실행 기록 확인 |

`True`는 조건을 만족한다는 뜻이고 `False`는 만족하지 않는다는 뜻입니다. 조건표에 `.sum()`을 적용하면 True인 행을 셀 수 있습니다. 예를 들어 본문 존재 여부를 표시한 조건표라면 그 합계는 본문이 있는 행의 수입니다.

#### 전체 흐름과 변수의 연결

1. 준비 셀: quotes_raw.csv를 읽어 원본 표인 `df`를 만듭니다.
2. 필수 1: df의 행 수·컬럼·자료형·일부 실제 값을 관찰합니다.
3. 필수 2: `check_df`라는 복사본에서 내용 없는 값과 중복 후보를 확인합니다.
4. 과제 1: 다시 원본 df의 각 컬럼에서 확인용 변수를 만들고 목적별 건수·비율을 계산합니다.
5. 해석: 현재 파일에서 확인한 결과와, 수집 기록이 있어야 판단할 결과를 구분합니다.

check_df의 표시가 바뀌어도 df나 CSV가 자동으로 바뀌는 것은 아닙니다. 과제 1의 `text`·`author`·`tags`·`source`도 원본에 다시 대입하지 않는 확인용 변수입니다. 이번 강의는 정제 파일을 만드는 단계가 아닙니다.


## 실습 준비

1. 앞선 API 실습의 `quotes_raw.csv`를 현재 작업 폴더에 준비하세요.
2. 아래 제공 코드를 실행하세요. 파일이 없다면 경로와 앞선 실습의 저장 여부를 확인하세요.
3. 필수 1 → 필수 2 → 과제 1 순서로 실행하세요. 각 문제는 준비 셀의 `df`, `expected_columns`를 사용합니다.
4. 실제 출력값으로 표를 채우세요. 수집한 시점이나 범위에 따라 건수는 달라질 수 있습니다.

교안 2·3절과 같이 `keep_default_na=False`로 읽습니다. 빈 값이나 `NaN`, `NULL` 같은 문자열을 기본 규칙으로 결측값으로 바꾸지 않고, 이후 빈 문자열과 공백을 직접 확인하기 위한 설정입니다. 이 설정이 누락된 정보를 채워 주는 것은 아닙니다.


In [1]:
from pathlib import Path
import pandas as pd

file_path = Path("quotes_raw.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "quotes_raw.csv 파일이 없습니다. 파일 경로와 2장 API 실습의 저장 결과를 확인하세요."
    )

df = pd.read_csv(
    file_path,
    encoding="utf-8-sig",
    keep_default_na=False
)

expected_columns = ["text", "author", "tags", "source_url"]
print("불러온 파일:", file_path)
print("전체 데이터 건수:", len(df))


불러온 파일: quotes_raw.csv
전체 데이터 건수: 100


### 준비 코드 상세 해설 — 파일에서 확인용 표까지

| 코드 | 하는 일 | 결과를 읽는 기준 |
|---|---|---|
| `from pathlib import Path` | 파일 경로를 다룰 도구를 가져옴 | 아직 파일을 읽지 않음 |
| `import pandas as pd` | 표를 다루는 pandas를 pd라는 이름으로 사용 | pd.read_csv() 같은 기능을 호출할 준비 |
| `file_path = Path("quotes_raw.csv")` | 입력 파일의 위치를 변수에 보관 | 현재 작업 폴더를 기준으로 찾음 |
| `file_path.exists()` | 해당 경로가 존재하는지 확인 | 내용·컬럼·품질 검사까지 수행하지 않음 |
| `if not ...` | 존재하지 않을 때 안내 분기로 들어감 | not은 True와 False를 반대로 바꿈 |
| `raise FileNotFoundError(...)` | 파일이 없다는 오류를 발생시킴 | 이 셀의 이후 읽기 코드는 실행되지 않음 |
| `pd.read_csv(...)` | CSV를 읽어 DataFrame 생성 | 성공한 결과를 df에 보관 |
| `encoding="utf-8-sig"` | 앞선 실습의 CSV 문자 인코딩에 맞춰 읽음 | 본문 내용이나 누락값을 보충하는 옵션은 아님 |
| `keep_default_na=False` | 기본 결측 문자열 자동 변환을 사용하지 않음 | 빈 문자열과 공백을 뒤에서 직접 확인 |
| `expected_columns = [...]` | 필요한 네 컬럼 이름을 List에 보관 | df에 새 컬럼을 추가하는 코드가 아님 |
| `len(df)` | 읽은 표의 행 수를 반환 | 사이트에 원래 존재하는 전체 건수와 구분 |

파일 경로가 존재해도 올바른 CSV라는 보장은 없습니다. 읽기에 실패했다면 오류를 확인한 뒤 입력 파일을 준비하고 준비 셀부터 다시 실행하세요. 반대로 컬럼 이름만 있고 데이터 행이 없는 CSV는 읽을 수 있지만 행 수가 0입니다. **내용이 전혀 없는 파일과, 헤더는 있지만 행이 없는 표는 다릅니다.**

`keep_default_na=False`로 읽은 문자 `"NaN"`·`"NULL"`과 실제 결측값은 다릅니다. 이번 코드가 빈 문자열로 판단하는 것은 공백 제거 후 내용이 없는 값입니다. 이 문자들이 있다는 이유만으로 자동으로 모두 누락 처리하지 않습니다.

노트북에서는 위 셀에서 만든 변수를 아래 셀이 사용합니다. df는 준비 셀에서 만들어지므로, 문제 셀만 먼저 실행하면 사용할 변수가 준비되지 않았을 수 있습니다. 입력 파일을 바꾸었을 때도 준비 셀부터 다시 실행해야 새 파일의 결과를 확인할 수 있습니다.


---

## 필수 1. 수집 데이터의 기본 구조 확인

### 문제 1-1. 예상한 구조와 실제 CSV 비교하기

#### 문제 설명

앞선 API 실습에서 저장한 CSV의 행 수, 컬럼, 자료형과 처음 일부 값을 확인하세요. 필요한 컬럼이 있는지와 예상하지 않은 컬럼이 있는지를 각각 비교하고, 기본 구조 확인만으로 알 수 없는 사항도 설명하세요.

#### 요구사항

1. `df.shape`, `len(df)`, `df.columns.tolist()`, `df.dtypes`, `df.info()`를 사용하여 구조를 출력하세요. `df.info()`는 직접 호출하세요.
2. `expected_columns` 중 실제 컬럼에 없는 항목을 `missing_columns`에, 실제 컬럼 중 예상 목록에 없는 항목을 `extra_columns`에 List로 저장하고 출력하세요.
3. `df.head(3)`으로 처음 3행의 실제 값을 확인하세요.
4. `source_url` 컬럼이 있을 때만 `value_counts(dropna=False)`로 출처별 건수를 출력하세요.
5. `df.empty`이면 데이터가 비어 있다는 안내를, 누락·추가 컬럼이 있으면 컬럼 확인이 필요하다는 안내를, 그 외에는 기본 행·컬럼 구조 확인 완료 안내를 출력하세요.
6. 구조 확인표를 작성하고 Q1~Q5에 답하세요. 문제가 발견되어도 이 셀에서 컬럼을 삭제·추가하거나 CSV를 덮어쓰지 마세요.

#### 해석 질문

**Q1.** 실제 `shape`는 무엇이며 두 숫자는 각각 무엇을 뜻하나요? 교안의 가정 결과 `(80, 4)`는 어떻게 해석하나요?

**Q2.** 실제 컬럼이 `["text", "author", "tags", "Unnamed: 0"]`이라면 누락·추가 컬럼은 무엇인가요? 컬럼 수가 4개이면 충분한가요?

**Q3.** `dtypes`, `info()`, `head(3)`는 각각 무엇을 확인하는 데 도움이 되나요? 처음 3행이 정상이라면 전체 품질도 정상인가요?

**Q4.** 교안처럼 예상 수집 건수가 100개인데 저장 건수가 70개라면 무엇을 확인해야 하나요? 100개로 일치하면 전체 수집 성공을 확정할 수 있나요?

**Q5.** 현재 파일의 기본 구조를 확인한 뒤에도 추가로 확인해야 할 품질 항목은 무엇인가요?

#### 제출 결과

- 구조 확인 코드와 전체 실행 결과
- 구조 확인표
- Q1~Q5 답변


### 풀이 전 개념과 접근 순서

먼저 “표의 크기”, “필요한 이름”, “실제 값”을 나누어 확인하세요. df.shape의 첫 숫자는 행 수, 두 번째 숫자는 컬럼 수입니다. 컬럼 수가 같아도 이름이 다를 수 있으므로 expected_columns와 실제 컬럼을 양쪽 방향으로 비교해야 합니다.

| 작성할 부분 | 생각할 순서 |
|---|---|
| missing_columns | 예상 목록에서 시작하여 실제 표에 없는 이름을 모음 |
| extra_columns | 실제 목록에서 시작하여 예상 목록에 없는 이름을 모음 |
| 자료형·미리보기 | dtypes·info()로 형태를 보고 head(3)으로 실제 내용 확인 |
| 출처 집계 | 컬럼이 있는지 확인한 뒤 같은 출처의 행 수를 셈 |
| 마지막 안내 | 비어 있는 표인지 → 컬럼 차이가 있는지 → 기본 구조가 맞는지 |

`df.info()`는 정보를 직접 출력하므로 그 자체를 호출하세요. 빈 문자열은 이번 읽기 설정에서 실제 결측 표시와 다르므로 info()만으로 내용 누락을 모두 판단하지 않습니다. head(3)은 일부 행의 미리보기이며 전체 검사를 대신하지 않습니다.

조건문의 출력이 “기본 구조 확인 완료”여도 값 누락·중복·수집 완료는 다음 확인이 필요합니다. 표에는 본인의 실행 결과를 적고 교안의 가정 숫자와 구분하세요.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과를 남기세요.


In [2]:
# 필수 1. 수집 데이터의 기본 구조 확인 코드를 작성하세요.
# 요구사항의 변수명을 사용하고 원본 데이터는 유지하세요.

print("Shape:", df.shape)
print("전체 행 수:", len(df))
print("컬럼 목록:", df.columns.tolist())
print("컬럼별 자료형:")
print(df.dtypes)
df.info()

missing_columns = [col for col in expected_columns if col not in df.columns]
extra_columns = [col for col in df.columns if col not in expected_columns]
print("누락 컬럼:", missing_columns)
print("추가 컬럼:", extra_columns)

print(df.head(3))

if "source_url" in df.columns:
    print("출처별 건수:")
    print(df["source_url"].value_counts(dropna=False))

if df.empty:
    print("데이터가 비어 있습니다.")
elif missing_columns or extra_columns:
    print("컬럼 확인이 필요합니다.")
else:
    print("기본 행·컬럼 구조 확인 완료")

Shape: (100, 4)
전체 행 수: 100
컬럼 목록: ['text', 'author', 'tags', 'source_url']
컬럼별 자료형:
text          str
author        str
tags          str
source_url    str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   text        100 non-null    str  
 1   author      100 non-null    str  
 2   tags        100 non-null    str  
 3   source_url  100 non-null    str  
dtypes: str(4)
memory usage: 3.3 KB
누락 컬럼: []
추가 컬럼: []
                                                text           author  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   

                                           tags  \
0        change, deep-thoughts, thinking, world   
1                            abilities, choices   
2

### 답변 작성란


| 항목 | 실제 결과 | 해석·확인할 점 |
|---|---|---|
| Shape / 전체 행 수 | (100, 4) / 100 | 명언 100개, 컬럼 4개 |
| 실제 컬럼 목록 | ['text', 'author', 'tags', 'source_url'] | 예상한 네 컬럼과 같음 |
| 누락 컬럼 / 추가 컬럼 | [] / [] | 빠지거나 더 들어간 컬럼 없음 |
| 컬럼별 자료형 | 네 컬럼 모두 object | 모두 문자열로 읽힘 |
| 처음 3행의 주요 값 | 1행 Albert Einstein, 2행 J.K. Rowling, 3행 Albert Einstein 【실행 후 확인】 | 본문·작성자·태그·출처가 제 칸에 들어 있음 |
| 출처별 건수 | page=1부터 page=10까지 각 10건 | 10페이지 모두에서 저장됨 |
| 기본 구조 판단 | 기본 행·컬럼 구조 확인 완료 | 값 누락·중복은 아직 확인 전 |


**Q1. 실제 `shape`는 무엇이며 두 숫자는 각각 무엇을 뜻하나요? 교안의 가정 결과 `(80, 4)`는 어떻게 해석하나요?**

- 답변: 실제 shape는 (100, 4)이고, 100은 행 수(명언 100개), 4는 컬럼 수입니다. 교안의 (80, 4)는 명언 80개가 4개 컬럼으로 저장되어 있다는 뜻이고, 이것만으로는 원래 수집해야 할 건수를 다 모았는지 알 수 없습니다.

**Q2. 실제 컬럼이 `["text", "author", "tags", "Unnamed: 0"]`이라면 누락·추가 컬럼은 무엇인가요? 컬럼 수가 4개이면 충분한가요?**

- 답변: 누락 컬럼은 source_url, 추가 컬럼은 Unnamed: 0입니다. Unnamed: 0은 인덱스가 저장된 흔적처럼 보입니다. 컬럼 수가 4개로 같아도 이름이 다르기 때문에 충분하지 않습니다.

**Q3. `dtypes`, `info()`, `head(3)`는 각각 무엇을 확인하는 데 도움이 되나요? 처음 3행이 정상이라면 전체 품질도 정상인가요?**

- 답변: dtypes는 각 컬럼의 자료형을, info()는 행 수·non-null 개수·자료형을 한 번에, head(3)은 실제 값의 모양을 확인하는 데 도움이 됩니다. head(3)은 처음 3행만 보는 미리보기라서, 3행이 정상이어도 전체 품질이 정상이라고 할 수는 없습니다.

**Q4. 교안처럼 예상 수집 건수가 100개인데 저장 건수가 70개라면 무엇을 확인해야 하나요? 100개로 일치하면 전체 수집 성공을 확정할 수 있나요?**

- 답변: 어느 페이지에서 빠졌는지 보려고 출처별 건수를 확인하고, 수집 때의 페이지별 기록과 오류 메시지, 종료 조건을 확인해야 합니다. 100개로 일치해도 중복이 섞이고 일부가 빠졌을 수 있고 정상 종료 기록도 필요해서, 건수만으로 전체 수집 성공을 확정할 수는 없습니다.

**Q5. 현재 파일의 기본 구조를 확인한 뒤에도 추가로 확인해야 할 품질 항목은 무엇인가요?**

- 답변: 각 컬럼 안의 값이 비어 있거나 공백만 있는지, 같은 본문·작성자가 반복된 중복 후보가 있는지, 수집 과정이 마지막 페이지까지 정상 완료됐는지를 추가로 확인해야 합니다.


---

## 필수 2. 중복·결측·출처 탐색

### 문제 2-1. 원본을 유지하며 품질 문제의 유형과 건수 확인하기

#### 문제 설명

교안의 확인용 복사본 방식으로 누락된 값과 중복 후보를 탐색하세요. 빈 문자열과 공백만 있는 문자열도 내용이 없는 값으로 확인합니다. 원본 `df`와 `quotes_raw.csv`는 유지하고, 중복 후보를 삭제하지 않습니다.

#### 요구사항

1. `expected_columns`를 기준으로 `missing_columns`를 다시 확인하세요. 필수 컬럼이 하나라도 없으면 `ValueError`로 안내하고 이후 탐색을 멈추세요.
2. `check_df = df.copy()`로 확인용 복사본을 만드세요.
3. 각 예상 컬럼에 `astype("string").str.strip()`을 적용하고, 빈 문자열인 위치를 `check_df.loc[...]`와 `pd.NA`로 표시하세요. 이 과정은 `check_df`에만 적용하세요.
4. `check_df[expected_columns].isna().sum()`으로 컬럼별 결측·빈 문자열 건수를 출력하세요.
5. `text`와 `author`가 모두 존재하는 조건을 `valid_mask`에 저장하세요. 이 조건과 `duplicated(subset=["text", "author"], keep=False)`를 `&`로 결합하여 `duplicate_mask`를 만드세요.
6. 중복 후보 행 수를 출력하고, `.loc[duplicate_mask, ["text", "author"]].head(10)`으로 후보의 본문·작성자를 확인하세요.
7. `check_df["source_url"].value_counts(dropna=False).sort_index()`로 출처별 건수를 출력하세요. 전체 행 수·본문 누락·작성자 누락·중복 후보 행 수를 요약하고 원본을 유지했음을 안내하세요.
8. 품질 확인표와 Q1~Q6 답변을 작성하세요. 중복 후보가 없으면 0건으로 기록하세요.

#### 해석 질문

**Q1.** `isna()`만으로 빈 문자열과 공백 문자열을 모두 찾기 어려운 이유는 무엇인가요? `keep_default_na=False`와 공백 제거는 각각 어떤 역할을 하나요?

**Q2.** `check_df`에서 빈 문자열을 `pd.NA`로 표시하는 것과 원본을 실제 정제하여 저장하는 것은 어떻게 다른가요?

**Q3.** 중복 기준에서 `source_url`을 제외한 이유와, 본문·작성자가 모두 있는 행만 비교하는 이유는 무엇인가요?

**Q4.** 교안의 네 행이 순서대로 `(명언 A, 작성자 1)`, `(명언 B, 작성자 2)`, `(명언 A, 작성자 1)`, `(명언 C, 작성자 3)`이면 중복 후보는 몇 번째 행이고 몇 개인가요? 이 숫자가 반드시 삭제할 행 수인가요?

**Q5.** 실제 실행에서 가장 많이 누락된 컬럼과 중복 후보 행 수는 무엇인가요? 0건이면 탐색을 잘못한 것인가요?

**Q6.** 출처별 건수만으로 수집하지 못한 페이지의 존재를 확정할 수 있나요? 본문 누락이나 중복 후보를 발견하면 먼저 무엇을 확인해야 하나요?

#### 제출 결과

- 결측·중복·출처 탐색 코드와 출력
- 품질 확인표
- Q1~Q6 답변


### 풀이 전 개념과 접근 순서

이번에는 **원본을 보관하면서 같은 기준으로 내용 없는 값을 세는 것**이 중요합니다. check_df를 만든 다음 네 컬럼에 같은 절차를 적용하세요.

1. 필요한 컬럼이 모두 있는지 확인합니다. 없으면 안내 오류를 내고 멈춥니다.
2. df.copy()로 복사본을 만듭니다.
3. 문자열의 양끝 공백을 제거하고, 빈 문자열인 칸을 복사본에서 pd.NA로 표시합니다.
4. isna()의 True/False를 컬럼별로 합해 누락 건수를 셉니다.
5. 본문·작성자가 모두 있는 조건과 두 값이 반복되는 조건을 결합합니다.
6. 후보 전체 건수와 후보 미리보기를 구분하여 기록합니다.

| 표현 | 풀이에 필요한 뜻 |
|---|---|
| `.loc[행 조건, 컬럼]` | 어떤 행의 어떤 컬럼인지 지정 |
| `.isna()` / `.notna()` | 결측인가 / 결측이 아닌가 |
| `&` | 같은 행에서 두 조건을 모두 만족하는지 확인 |
| `subset` | 중복을 비교할 컬럼 지정 |
| `keep=False` | 같은 조합의 첫 행까지 모두 후보로 표시 |
| `.sum()` | 조건표에서 True인 개수를 셈 |
| `dropna=False` | 출처 집계에서 실제 결측도 포함 |

한 행에서 여러 필드가 비어 있으면 컬럼별 누락 집계에 각각 포함됩니다. 중복 후보 수는 같은 조합에 속한 모든 행 수이며, 삭제할 행 수는 아닙니다. 공백 표시를 바꿀 대상은 check_df인지, 원본 df인지 확인하고 작성하세요.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과를 남기세요.


In [3]:
# 필수 2. 중복·결측·출처 탐색 코드를 작성하세요.
# 요구사항의 변수명을 사용하고 원본 데이터는 유지하세요.

missing_columns = [col for col in expected_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")

check_df = df.copy()

for col in expected_columns:
    check_df[col] = check_df[col].astype("string").str.strip()
    check_df.loc[check_df[col] == "", col] = pd.NA

missing_counts = check_df[expected_columns].isna().sum()
print("컬럼별 결측·빈 문자열 건수:")
print(missing_counts)

valid_mask = check_df["text"].notna() & check_df["author"].notna()
duplicate_mask = valid_mask & check_df.duplicated(subset=["text", "author"], keep=False)

print("중복 후보 행 수:", duplicate_mask.sum())
print(check_df.loc[duplicate_mask, ["text", "author"]].head(10))

print("출처별 건수:")
print(check_df["source_url"].value_counts(dropna=False).sort_index())

print("전체 행 수:", len(check_df))
print("본문 누락:", missing_counts["text"])
print("작성자 누락:", missing_counts["author"])
print("중복 후보 행 수:", duplicate_mask.sum())
print("원본 df와 quotes_raw.csv는 변경하지 않았습니다.")

컬럼별 결측·빈 문자열 건수:
text          0
author        0
tags          3
source_url    0
dtype: int64
중복 후보 행 수: 0
Empty DataFrame
Columns: [text, author]
Index: []
출처별 건수:
source_url
https://quotes.toscrape.com/api/quotes?page=1     10
https://quotes.toscrape.com/api/quotes?page=10    10
https://quotes.toscrape.com/api/quotes?page=2     10
https://quotes.toscrape.com/api/quotes?page=3     10
https://quotes.toscrape.com/api/quotes?page=4     10
https://quotes.toscrape.com/api/quotes?page=5     10
https://quotes.toscrape.com/api/quotes?page=6     10
https://quotes.toscrape.com/api/quotes?page=7     10
https://quotes.toscrape.com/api/quotes?page=8     10
https://quotes.toscrape.com/api/quotes?page=9     10
Name: count, dtype: Int64
전체 행 수: 100
본문 누락: 0
작성자 누락: 0
중복 후보 행 수: 0
원본 df와 quotes_raw.csv는 변경하지 않았습니다.


### 답변 작성란


| 컬럼 | 결측·빈 문자열 건수 | 확인할 점 |
|---|---|---|
| text | 0 | 본문 분석에 쓸 수 없는 행 없음 |
| author | 0 | 작성자별 분석에 문제 없음 |
| tags | 【실행 후 확인】 | 태그가 없는 명언이 있으면 태그별 분석에서만 제한 대상 |
| source_url | 0 | 모든 행의 출처가 기록됨 |

| 항목 | 실제 결과 |
|---|---|
| 전체 행 수 | 100 |
| 중복 후보 행 수 | 0 |
| 중복 후보 본문·작성자 확인 결과 | 후보 없음(빈 표 출력) |
| 출처별 건수 | page=1부터 page=10까지 각 10건 |
| 추가로 확인할 수집 기록 | 2장 1강 실행에서 마지막 페이지의 has_next가 False였는지, 중간에 오류가 없었는지 |


**Q1. `isna()`만으로 빈 문자열과 공백 문자열을 모두 찾기 어려운 이유는 무엇인가요? `keep_default_na=False`와 공백 제거는 각각 어떤 역할을 하나요?**

- 답변: keep_default_na=False로 읽으면 빈 칸이 결측값이 아니라 빈 문자열로 남아서 isna()가 찾지 못하고, 공백만 있는 문자열도 글자가 있는 것으로 취급됩니다. keep_default_na=False는 pandas가 마음대로 결측으로 바꾸지 않게 해서 원래 모습을 확인하게 해 주고, 공백 제거는 공백만 있는 값을 빈 문자열로 만들어 함께 찾을 수 있게 해 줍니다.

**Q2. `check_df`에서 빈 문자열을 `pd.NA`로 표시하는 것과 원본을 실제 정제하여 저장하는 것은 어떻게 다른가요?**

- 답변: check_df에서 pd.NA로 표시하는 건 세기 위해 복사본에만 표시한 것이라 원본 df와 CSV는 그대로입니다. 실제 정제는 기준에 따라 값을 바꾸거나 행을 지운 결과를 새 파일로 저장하는 거라서 다릅니다.

**Q3. 중복 기준에서 `source_url`을 제외한 이유와, 본문·작성자가 모두 있는 행만 비교하는 이유는 무엇인가요?**

- 답변: source_url은 같은 명언이 다른 페이지에서 반복 수집되면 값이 달라질 수 있어서, 포함하면 진짜 중복을 놓칠 수 있기 때문에 제외했습니다. 본문이나 작성자가 빈 행끼리는 비어 있다는 이유만으로 같다고 묶일 수 있어서 둘 다 있는 행만 비교합니다.

**Q4. 교안의 네 행이 순서대로 `(명언 A, 작성자 1)`, `(명언 B, 작성자 2)`, `(명언 A, 작성자 1)`, `(명언 C, 작성자 3)`이면 중복 후보는 몇 번째 행이고 몇 개인가요? 이 숫자가 반드시 삭제할 행 수인가요?**

- 답변: 1번째와 3번째 행이 중복 후보이고 2개입니다. keep=False라서 첫 행까지 포함된 숫자이고, 하나를 남기면 실제로 지울 행은 1개라서 후보 수를 그대로 삭제할 행 수로 보면 안 됩니다.

**Q5. 실제 실행에서 가장 많이 누락된 컬럼과 중복 후보 행 수는 무엇인가요? 0건이면 탐색을 잘못한 것인가요?**

- 답변: 본문·작성자·출처 누락은 0건이고, 누락이 있다면 tags가 가장 많을 것 같습니다 【실행 후 확인】. 중복 후보는 0건입니다. 0건이라도 탐색을 잘못한 게 아니라 이번 기준으로 문제가 없었다는 결과를 그대로 기록하면 됩니다.

**Q6. 출처별 건수만으로 수집하지 못한 페이지의 존재를 확정할 수 있나요? 본문 누락이나 중복 후보를 발견하면 먼저 무엇을 확인해야 하나요?**

- 답변: 확정할 수 없습니다. 출처별 건수는 저장된 행을 센 것일 뿐이라, 원래 몇 페이지까지 있었는지나 요청 실패 기록은 알 수 없습니다. 본문 누락이나 중복 후보를 발견하면 먼저 해당 행의 source_url과 원본 페이지, 수집 기록을 확인하겠습니다.


---

## 과제 1. 분석 목적별 활용 가능성 판단

### 문제 3-1. 필요한 필드의 존재 여부와 수집 완료 근거 구분하기

#### 문제 설명

필수 실습과 같은 원본 `df`를 사용하여 분석 목적별 활용 가능 건수와 비율을 확인하세요. 교안 3절의 값 존재 여부 확인 코드를 적용하는 기본 수준의 과제입니다. 실제 텍스트 분석이나 데이터 삭제는 수행하지 않습니다. 아래 교안 사례는 표를 읽고 답합니다.


**교안의 가정 사례 A — 분석 가능 건수**

아래 표는 실제 CSV가 아니라 교안 연습문제 3의 여섯 행입니다. 빈칸은 내용이 없는 값입니다. 이 표를 원본 CSV에 추가하지 마세요.

| 행 | text | author | tags |
|---|---|---|---|
| 1 | 명언 A | 작성자 A | life |
| 2 | 명언 A | 작성자 A | life |
| 3 | 명언 B |  | learning |
| 4 | 명언 C | 작성자 C |  |
| 5 |  | 작성자 D | thinking |
| 6 | 명언 D | 작성자 E | success |

**교안의 가정 사례 B — 페이지별 수집 결과**

| 페이지 | 요청 결과 | 수집 건수 |
|---|---|---|
| 1 | 성공 | 10 |
| 2 | 성공 | 10 |
| 3 | 실패 | 0 |


#### 요구사항

1. `required_columns = ["text", "author", "tags", "source_url"]`를 만들고 누락 컬럼이 있으면 `ValueError`로 멈추세요.
2. 원본의 각 컬럼에 `fillna("").astype(str).str.strip()`을 적용한 결과를 각각 `text`, `author`, `tags`, `source` 변수에 저장하세요. `df`의 컬럼에 다시 대입하지 마세요.
3. 각 변수에 `.ne("")`를 적용하여 `has_text`, `has_author`, `has_tags`, `has_source`를 만드세요.
4. 본문 존재 건수를 `text_count`, 본문·작성자가 모두 있는 건수를 `author_count`, 본문·태그가 모두 있는 건수를 `tag_count`로 계산하세요.
5. `text`와 `author`로 `duplicate_keys` DataFrame을 만들고, 두 값이 모두 있는 조건과 `duplicated(subset=["text", "author"], keep=False)`를 결합하여 `duplicate_mask`를 만드세요.
6. 전체 행 수·세 분석 목적별 건수·중복 후보 행 수·`(~has_source).sum()`으로 계산한 출처 누락 건수를 출력하세요.
7. `len(df) > 0`일 때 각 활용 가능 건수를 전체 행 수로 나눈 비율을 `:.1%` 형식으로 출력하세요. 비어 있으면 수집된 데이터가 없다고 안내하세요.
8. 실제 결과표와 수집 실패 확인표를 작성하고 Q1~Q6에 답하세요. 사례 A의 중복을 하나만 남겼을 때의 건수는 표로 판단하고 제거 코드를 작성하지 마세요.

#### 해석 질문

**Q1.** 단어 빈도·작성자별·태그별 분석에 각각 어떤 필드가 필요한가요? 작성자나 태그가 없다는 이유로 본문도 활용할 수 없다고 판단해도 되나요?

**Q2.** 실제 전체 수집 건수와 세 목적별 활용 가능 건수·비율은 얼마인가요? 이 값이 최종 정제 후 사용할 건수인가요?

**Q3.** 사례 A의 전체 행·본문 존재·본문과 작성자 존재·본문과 태그 존재 건수는 각각 얼마인가요? 중복 후보는 어떤 행이며, 본문이 있는 데이터에서 같은 명언을 하나만 남긴다면 몇 개가 되나요?

**Q4.** 사례 B에서 확보한 명언은 몇 개이며 전체 수집 완료로 판단할 수 있나요? 실패한 요청과 정상 종료 여부를 확인하려면 어떤 정보가 필요한가요?

**Q5.** CSV에 80행이 있고 본문·작성자에 결측이 없지만 일부 페이지의 요청 기록을 확인하지 못했다면 전체 수집 완료를 확정할 수 있나요? API와 HTML의 정상 종료 근거는 각각 무엇인가요?

**Q6.** 중복이 단어 빈도에 줄 수 있는 영향은 무엇인가요? 본문·작성자·태그 누락, 중복 후보, 출처 누락, 페이지 실패를 확인했다면 다음 단계에서 각각 무엇을 검토해야 하나요?

#### 제출 결과

- 분석 목적별 집계 코드와 출력
- 실제 결과표·수집 실패 확인표
- 사례 A·B를 포함한 Q1~Q6 답변


### 풀이 전 개념과 접근 순서

과제는 “빈칸이 몇 개인가?”에서 한 걸음 더 나아가 **“이 분석에 필요한 값이 있는 행은 몇 개인가?”**를 묻습니다. 필수 2의 복사본 대신 원본 df에서 확인용 문자열 변수를 새로 만드세요.

| 단계 | 생각할 질문 |
|---|---|
| fillna("") | 실제 결측을 확인용 빈 문자열로 통일했는가? |
| astype(str).str.strip() | 문자열 양끝 공백까지 확인했는가? |
| ne("") | 내용이 있는 행을 True로 표시했는가? |
| 조건 결합 | 작성자별·태그별 집계에 본문 존재도 함께 포함했는가? |
| 중복 후보 | 본문·작성자가 모두 있는 조건을 포함했는가? |
| 비율 | 전체 행 수가 0인지 먼저 확인했는가? |

author_count는 작성자 이름의 종류 수가 아니라 조건에 맞는 **명언 행 수**입니다. tag_count도 태그 단어 수가 아닌 행 수입니다. 출처 존재 여부는 별도로 집계하며, 요구사항의 세 목적별 건수 조건에 임의로 새 조건을 추가하지 않습니다.

사례 A는 각 행의 본문·작성자·태그 존재 여부를 True/False로 적어 본 뒤 필요한 조건이 모두 참인 행을 세면 됩니다. 세 목적에 같은 행이 포함될 수 있으므로 결과를 서로 더하지 마세요. 비율의 분모는 각 목적에서 선택된 행 수가 아니라 전체 행 수입니다.

사례 B는 “이미 저장한 건수”와 “전체 수집이 끝났는지”를 나누어 답하세요. 실제 수집 기록을 확인하지 못했다면 완료 여부는 미확인으로 남기고, 필요한 기록을 설명하세요.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과를 남기세요.


In [4]:
# 과제 1. 분석 목적별 활용 가능성 판단 코드를 작성하세요.
# 요구사항의 변수명을 사용하고 원본 데이터는 유지하세요.

required_columns = ["text", "author", "tags", "source_url"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")

text = df["text"].fillna("").astype(str).str.strip()
author = df["author"].fillna("").astype(str).str.strip()
tags = df["tags"].fillna("").astype(str).str.strip()
source = df["source_url"].fillna("").astype(str).str.strip()

has_text = text.ne("")
has_author = author.ne("")
has_tags = tags.ne("")
has_source = source.ne("")

text_count = has_text.sum()
author_count = (has_text & has_author).sum()
tag_count = (has_text & has_tags).sum()

duplicate_keys = pd.DataFrame({"text": text, "author": author})
duplicate_mask = (
    has_text
    & has_author
    & duplicate_keys.duplicated(subset=["text", "author"], keep=False)
)

print("전체 행 수:", len(df))
print("단어 빈도 분석 가능 건수:", text_count)
print("작성자별 분석 가능 건수:", author_count)
print("태그별 분석 가능 건수:", tag_count)
print("중복 후보 행 수:", duplicate_mask.sum())
print("출처 누락 건수:", (~has_source).sum())

if len(df) > 0:
    total = len(df)
    print(f"단어 빈도 분석 가능 비율: {text_count / total:.1%}")
    print(f"작성자별 분석 가능 비율: {author_count / total:.1%}")
    print(f"태그별 분석 가능 비율: {tag_count / total:.1%}")
else:
    print("수집된 데이터가 없습니다.")

전체 행 수: 100
단어 빈도 분석 가능 건수: 100
작성자별 분석 가능 건수: 100
태그별 분석 가능 건수: 97
중복 후보 행 수: 0
출처 누락 건수: 0
단어 빈도 분석 가능 비율: 100.0%
작성자별 분석 가능 비율: 100.0%
태그별 분석 가능 비율: 97.0%


### 답변 작성란


| 분석 목적 | 필요한 필드 | 실제 활용 가능 건수 | 전체 행 수 대비 비율 |
|---|---|---|---|
| 단어 빈도 분석 | text | 100 | 100.0% |
| 작성자별 명언 분석 | text, author | 100 | 100.0% |
| 태그별 명언 분석 | text, tags | 【실행 후 확인】 | 【실행 후 확인】 |

- 전체 행 수: 100
- 중복 후보 행 수: 0
- 출처 누락 건수: 0
- 위 건수·비율로 판단할 수 없는 사항: 사이트의 모든 페이지를 빠짐없이 수집했는지, 값의 내용이 정확한지(빈칸이 아니라는 것만 확인함)

| 수집 실패·누락 상황 | 우선 확인할 정보 |
|---|---|
| HTTP 요청 실패 | 실패한 페이지의 요청 URL, 상태 코드, 오류 메시지 |
| JSON 파싱 실패 | 응답의 Content-Type과 응답 본문이 JSON 형식인지 |
| HTML 요소 누락 | 해당 페이지 HTML 구조와 선택자의 태그·클래스 이름 |
| 중간 페이지 누락 | 페이지별 처리 기록, source_url별 건수, 종료 조건(has_next 또는 Next 링크) |
| 필수 필드 누락 | 해당 행의 source_url과 원본 페이지에 실제 값이 있는지 |


**Q1. 단어 빈도·작성자별·태그별 분석에 각각 어떤 필드가 필요한가요? 작성자나 태그가 없다는 이유로 본문도 활용할 수 없다고 판단해도 되나요?**

- 답변: 단어 빈도 분석에는 text가, 작성자별 분석에는 text와 author가, 태그별 분석에는 text와 tags가 필요합니다. 작성자나 태그가 없어도 본문이 있으면 단어 빈도 분석에는 쓸 수 있어서 본문까지 못 쓴다고 판단하면 안 됩니다.

**Q2. 실제 전체 수집 건수와 세 목적별 활용 가능 건수·비율은 얼마인가요? 이 값이 최종 정제 후 사용할 건수인가요?**

- 답변: 전체 100건이고, 단어 빈도 분석 100건(100.0%), 작성자별 분석 100건(100.0%), 태그별 분석은 【실행 후 확인】입니다. 이 값은 필요한 필드가 있는지만 본 것이라 중복 처리 등을 거친 최종 정제 후 건수는 아닙니다.

**Q3. 사례 A의 전체 행·본문 존재·본문과 작성자 존재·본문과 태그 존재 건수는 각각 얼마인가요? 중복 후보는 어떤 행이며, 본문이 있는 데이터에서 같은 명언을 하나만 남긴다면 몇 개가 되나요?**

- 답변: 전체 6행, 본문 존재 5건(1, 2, 3, 4, 6행), 본문과 작성자 존재 4건(1, 2, 4, 6행), 본문과 태그 존재 4건(1, 2, 3, 6행)입니다. 중복 후보는 본문·작성자가 같은 1행과 2행이고, 본문이 있는 5건에서 같은 명언을 하나만 남기면 4개가 됩니다.

**Q4. 사례 B에서 확보한 명언은 몇 개이며 전체 수집 완료로 판단할 수 있나요? 실패한 요청과 정상 종료 여부를 확인하려면 어떤 정보가 필요한가요?**

- 답변: 확보한 명언은 20개입니다. 3페이지 요청이 실패했기 때문에 전체 수집 완료라고 판단할 수 없습니다. 실패 원인을 알려면 3페이지의 상태 코드와 오류 메시지가, 정상 종료 여부를 알려면 마지막 페이지에서 종료 조건에 도달했다는 기록이 필요합니다.

**Q5. CSV에 80행이 있고 본문·작성자에 결측이 없지만 일부 페이지의 요청 기록을 확인하지 못했다면 전체 수집 완료를 확정할 수 있나요? API와 HTML의 정상 종료 근거는 각각 무엇인가요?**

- 답변: 확정할 수 없습니다. 결측이 없다는 건 저장된 80행의 상태일 뿐이고, 요청하지 못한 페이지가 있었는지는 기록을 봐야 알 수 있습니다. API는 마지막 응답의 has_next가 False인 것, HTML은 마지막 페이지에 Next 링크가 없는 것이 정상 종료 근거입니다.

**Q6. 중복이 단어 빈도에 줄 수 있는 영향은 무엇인가요? 본문·작성자·태그 누락, 중복 후보, 출처 누락, 페이지 실패를 확인했다면 다음 단계에서 각각 무엇을 검토해야 하나요?**

- 답변: 중복이 있으면 같은 명언의 단어가 두 번 세어져 단어 빈도가 실제보다 높게 나올 수 있습니다. 다음 단계에서 본문 누락 행은 제외할지, 작성자·태그 누락 행은 해당 분석에서만 제한할지, 중복 후보는 실제 반복 수집인지 확인한 뒤 하나만 남길지 검토하겠습니다. 출처 누락은 수집 기록을 확인하고, 페이지 실패는 다시 수집이 필요한지 검토하겠습니다.


## 개념과 결과를 함께 정리하기

| 구분할 것 | 이번 실습에서 기억할 기준 |
|---|---|
| 컬럼 존재 / 값 존재 | 이름이 있는지 먼저 확인하고, 각 행의 내용을 확인 |
| 결측 표시 / 빈 문자열 / 공백 문자열 | 표현이 다르므로 공백과 빈 문자열을 함께 탐색 |
| 원본 / 확인용 결과 | df·CSV는 유지하고 check_df·별도 변수에서 검사 |
| 누락된 칸의 수 / 누락된 행의 수 | 같은 행이 여러 컬럼의 누락 집계에 포함될 수 있음 |
| 중복 후보 행 수 / 삭제할 행 수 | keep=False는 첫 행까지 포함하므로 같은 숫자가 아님 |
| 후보 미리보기 / 전체 후보 집계 | head(10)은 표시 범위, sum()은 전체 조건의 합계 |
| 값 존재 / 내용의 적절성 | 빈칸이 아니라는 사실만으로 내용의 정확성을 보장하지 않음 |
| 활용 가능 비율 / 최종 분석 적합성 | 현재 비율은 필요한 필드의 존재만 판단 |
| 저장된 데이터의 상태 / 전체 수집 완료 | CSV의 품질과 페이지별 수집 기록을 함께 확인 |

결과를 설명할 때는 **무엇을 기준으로 → 몇 행을 확인했는지 → 어떤 결과가 나왔는지 → 무엇은 아직 확인하지 못했는지**를 연결하여 쓰세요. 실제 파일에서 0건이 나오면 그 기준에서 0건이라고 기록하면 됩니다. 교안의 가정 사례를 실제 파일에 섞어 문제를 만들 필요는 없습니다.


---

## 실습 마무리

**Q1. 이번 실습의 데이터 품질 확인 흐름은 무엇인가요?**

- 답변: 기본 구조(행 수·컬럼·자료형) 확인 → 복사본에서 결측·빈 문자열·공백 탐색 → 본문·작성자 기준 중복 후보 탐색 → 출처별 건수 확인 → 분석 목적별 활용 가능 건수·비율 계산 → 수집 완료 여부를 기록으로 따로 판단하는 흐름이었습니다.

**Q2. 컬럼이 존재하는 것과 필요한 값이 존재하는 것은 어떻게 다른가요?**

- 답변: 컬럼이 있다는 건 표에 그 이름이 있다는 것이고, 값이 있다는 건 각 행의 그 칸에 실제 내용이 있다는 뜻입니다. 컬럼이 있어도 일부 행은 비어 있을 수 있습니다.

**Q3. 중복 후보 행 수를 그대로 삭제할 행 수로 해석하면 안 되는 이유는 무엇인가요?**

- 답변: keep=False는 같은 조합의 첫 행까지 모두 후보로 세기 때문에, 하나를 남긴다면 삭제할 행은 후보 수보다 적습니다.

**Q4. 전체 수집 건수와 분석 목적별 활용 가능 건수가 달라질 수 있는 이유는 무엇인가요?**

- 답변: 분석마다 필요한 필드가 달라서, 그 필드가 비어 있는 행은 해당 분석에서만 빠지기 때문입니다.

**Q5. 현재 CSV만으로 알 수 없는 수집 과정의 정보는 무엇이며 어떻게 확인해야 하나요?**

- 답변: 페이지별 요청 성공·실패, 상태 코드, 오류 메시지, 마지막 페이지에서의 종료 조건 도달 여부는 CSV만으로 알 수 없습니다. 수집 코드의 실행 출력이나 기록을 확인해야 합니다.

**Q6. 이번 실습에서 원본은 어떻게 유지했고, 다음 정제 단계에는 어떤 판단 자료를 넘길 수 있나요?**

- 답변: check_df와 별도 변수에서만 확인하고 df와 quotes_raw.csv는 건드리지 않았습니다. 다음 정제 단계에는 컬럼별 누락 건수, 중복 후보 행, 분석 목적별 활용 가능 건수, 더 확인해야 할 수집 기록을 판단 자료로 넘길 수 있습니다.


### 제출 전 확인

- 필수 1·2와 과제 1의 코드·출력·표를 작성하세요.
- 문제별 해석 질문과 실습 마무리 질문에 빠짐없이 답하세요.
- 실제 수집 데이터와 교안의 가정 사례를 구분하세요.
- 중복·누락이 0건인 경우에도 실제 결과를 그대로 기록하세요.
- 원본 `df`와 `quotes_raw.csv`를 변경하거나 정제 파일을 생성하지 않았는지 확인하세요.
- 수집 기록을 확인하지 못한 경우 전체 완료를 확정하지 마세요.
- 코드·출력·답변이 포함된 `.ipynb`를 제출하세요. 새로운 CSV 제출은 요구하지 않습니다.
